# Sports & Leisure Revenue Decline Analysis
**Period:** 2018-07 to 2018-26  
**Dataset:** Olist E-Commerce (Brazil)  
**Goal:** Identify root causes of 51% revenue decline in Sports & Leisure category

## 1. Setup

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import openpyxl

pio.renderers.default = 'notebook_connected'

## 2. Load Data

In [10]:
xl = pd.ExcelFile(r'..\olist_dataset_clean.xlsx')

df_order_items = xl.parse('fact_order_items')
df_reviews = xl.parse('order_reviews')
df_customers = xl.parse('customers')
df_sellers = xl.parse('sellers')
df_products = xl.parse('products')
df_date = xl.parse('dim_date')


In [15]:
print('Duplicates check:')
print(f'  order_items:        {df_order_items.duplicated(subset="order_id").sum()}')
print(f'  products:           {df_products.duplicated(subset="product_key").sum()}')
print(f'  reviews:            {df_reviews.duplicated(subset="order_id").sum()}')
print(f'  sellers:            {df_sellers.duplicated(subset="seller_key").sum()}')
print(f'  customers:          {df_customers.duplicated(subset="customer_key").sum()}')

Duplicates check:
  order_items:        13984
  products:           0
  reviews:            0
  sellers:            0
  customers:          0


## 3. Build Base DataFrame

In [11]:
df_base = (
    df_order_items
    .merge(df_products, on='product_key')
    .merge(df_sellers, on='seller_key')
    .merge(df_customers, on='customer_key')
    .merge(df_date, left_on='order_date_key', right_on='Date', how='left')
    .merge(df_reviews, on='order_id', how='left')
)
# Filter: delivered orders only, exclude last 2 incomplete months
starting_month = df_base['Date'].dt.to_period('M').max() - 2
df_base = df_base[
    (df_base['Date'].dt.to_period('M') <= starting_month) &
    (df_base['order_status'] == 'delivered')
].copy()

In [20]:
sl = df_base[df_base['product_category_name_english'] == 'sports_leisure'].copy()

# Price buckets (Procentiliai - paliekam tavo gerą idėją)
low  = sl['price'].quantile(0.33)
high = sl['price'].quantile(0.66)
print(f'Price thresholds: low <= {low:.2f}, mid <= {high:.2f}, high > {high:.2f}')

def price_bucket(x):
    if x <= low:   return 'low'
    elif x <= high: return 'mid'
    else:           return 'high'

sl['price_category'] = sl['price'].apply(price_bucket)
sl['Delivery_Time']  = pd.to_numeric(sl['Delivery_Time'], errors='coerce')

Price thresholds: low <= 54.90, mid <= 109.95, high > 109.95


In [28]:
# 1. Filtruojame tik Sports & Leisure kategoriją
sl = df_base[
    df_base["product_category_name_english"] == "sports_leisure"
].copy()

# Datos stulpelio sutvarkymas saugumo dėlei
sl["Date"] = pd.to_datetime(sl["Date"])

# 2. Kainų segmentai naudojant procentilius (tavo originali idėja)
low = sl["price"].quantile(0.33)
high = sl["price"].quantile(0.66)
print(
    f"Price thresholds: low <= {low:.2f}, mid <= {high:.2f}, high > {high:.2f}"
)


def price_bucket(x):
    if x <= low:
        return "low"
    elif x <= high:
        return "mid"
    else:
        return "high"


sl["price_category"] = sl["price"].apply(price_bucket)
sl["Delivery_Time"] = pd.to_numeric(sl["Delivery_Time"], errors="coerce")

# 3. Savaitinė agregacija (visai istorijai, kad nesugadintume slenkančio vidurkio)
sl_export = (
    sl.groupby("Year Week")
    .agg(
        revenue=("price", "sum"),
        orders=("order_id", "nunique"),  # PATAISYTA: iš count į nunique
        avg_review=("review_score", "mean"),
        avg_delivery=("Delivery_Time", "mean"),
        avg_freight_ratio=("freight_ratio", "mean"),
        unique_customers=("customer_unique_id", "nunique"),
        unique_sellers=("seller_id", "nunique"),
    )
    .sort_index()
)

# Teisingas AOV (Average Order Value) skaičiavimas
sl_export["aov"] = sl_export["revenue"] / sl_export["orders"]

# 4. Kainų segmentų pajamos
price_segments = (
    sl.groupby(["Year Week", "price_category"])["price"]
    .sum()
    .unstack()
    .fillna(0)
)
# Užtikrinam abėcėlinę stulpelių tvarką prieš pervadinant
price_segments = price_segments[["high", "low", "mid"]]
price_segments.columns = ["revenue_high", "revenue_low", "revenue_mid"]
sl_export = sl_export.join(price_segments)

# 5. Pardavėjų statuso logika (New vs Returning) pagal tavo periodus
pre_s = sl[sl["Year Week"] < "2018-07"]
dur_s = sl[(sl["Year Week"] >= "2018-07") & (sl["Year Week"] <= "2018-26")]

pre_seller_ids = set(pre_s["seller_id"].unique())
dur_seller_ids = set(dur_s["seller_id"].unique())
returning_sellers = pre_seller_ids & dur_seller_ids


def seller_status(seller_id):
    if seller_id in returning_sellers:
        return "returning"
    else:
        return "new"


sl["seller_status"] = sl["seller_id"].apply(seller_status)

# Pardavėjų pajamų agregacija
seller_weekly = (
    sl.groupby(["Year Week", "seller_status"])["price"]
    .sum()
    .unstack()
    .fillna(0)
)
# Užsitikiname, kad pasiimame tik reikalingus stulpelius
seller_weekly = (
    seller_weekly[["new", "returning"]]
    if "new" in seller_weekly.columns
    else seller_weekly
)
seller_weekly.columns = ["revenue_new_sellers", "revenue_returning_sellers"]
sl_export = sl_export.join(seller_weekly)

# 6. 6 savaičių slenkančių vidurkių skaičiavimas (ant pilnų duomenų)
cols_to_roll = [
    "revenue",
    "orders",
    "aov",
    "avg_review",
    "avg_delivery",
    "avg_freight_ratio",
    "unique_customers",
    "unique_sellers",
    "revenue_high",
    "revenue_low",
    "revenue_mid",
    "revenue_new_sellers",
    "revenue_returning_sellers",
]

for col in cols_to_roll:
    sl_export[f"{col}_6w"] = (
        sl_export[col].rolling(window=6, min_periods=1).mean()
    )

# 7. FILTRAS EKSPORTUI: pasiliekame tik analizės periodą (išlaikant tikslų rolling_6w)
sl_export_final = sl_export.loc["2018-07":"2018-26"].copy()

# Pridedame Sort Week stulpelį, skirtą tvarkingam rikiavimui Power BI
sl_export_final["Sort Week"] = (
    sl_export_final.index.str.replace("-", "").astype(int)
)

# Pakeičiam decimal iš '.' į ','
sl_export_final.to_csv("sports_leisure_weekly.csv", index=True, decimal=",", sep=";")

print("Eksportas baigtas! Failas 'sports_leisure_weekly.csv' paruoštas Power BI.")
sl_export_final.head()

Price thresholds: low <= 54.90, mid <= 109.95, high > 109.95
Eksportas baigtas! Failas 'sports_leisure_weekly.csv' paruoštas Power BI.


,revenue,orders,avg_review,avg_delivery,avg_freight_ratio,unique_customers,unique_sellers,aov,revenue_high,revenue_low,...,avg_delivery_6w,avg_freight_ratio_6w,unique_customers_6w,unique_sellers_6w,revenue_high_6w,revenue_low_6w,revenue_mid_6w,revenue_new_sellers_6w,revenue_returning_sellers_6w,Sort Week
Year Week,,,,,,,,,,,,,,,,,,,,,
2018-07,17234.08,135,4.141379,16.373368,0.267820,129,59,127.659852,12560.77,1322.78,...,14.917072,0.264104,135.166667,62.166667,14735.730000,1534.915000,4123.498333,1362.016667,19032.126667,201807
2018-08,15285.65,147,3.418182,17.429560,0.285235,146,56,103.984014,7882.10,1877.80,...,15.614938,0.261255,136.000000,61.166667,13105.305000,1635.866667,4141.476667,1130.568333,17752.080000,201808
2018-09,20977.93,168,3.606557,17.662801,0.266914,166,74,124.868631,14604.15,2094.64,...,16.220303,0.264366,140.333333,61.833333,13361.515000,1747.191667,4148.170000,1256.146667,18000.730000,201809
2018-10,24871.30,173,3.603352,17.645091,0.255353,171,69,143.764740,18669.33,1763.24,...,16.897527,0.261999,149.000000,62.333333,13678.331667,1774.470000,4415.336667,1040.923333,18827.215000,201810
2018-11,18439.68,147,3.824561,15.271912,0.296309,145,77,125.440000,12484.23,2120.64,...,16.724449,0.266495,148.500000,64.166667,13601.791667,1784.643333,4392.738333,1228.805000,18550.368333,201811


## 5. Sports & Leisure — Weekly Export for Power BI

In [ ]:
print(sl['price_category'].value_counts())
print(price_segments)

In [15]:
sl = df_base[df_base['product_category_name_english'] == 'sports_leisure'].copy()

# Price buckets
low  = sl['price'].quantile(0.33)
high = sl['price'].quantile(0.66)
print(f'Price thresholds: low <= {low:.2f}, mid <= {high:.2f}, high > {high:.2f}')

def price_bucket(x):
    if x <= low:   return 'low'
    elif x <= high: return 'mid'
    else:           return 'high'

sl['price_category'] = sl['price'].apply(price_bucket)
sl['Delivery_Time']  = pd.to_numeric(sl['Delivery_Time'], errors='coerce')

# Weekly aggregation
sl_export = sl.groupby('Year Week').agg(
    revenue              = ('price', 'sum'),
    orders               = ('order_id', 'count'),
    avg_review           = ('review_score_clean', 'mean'),
    avg_delivery         = ('Delivery_Time', 'mean'),
    avg_freight_ratio    = ('freight_ratio', 'mean'),
    unique_customers     = ('customer_unique_id', 'nunique'),
    unique_sellers       = ('seller_id', 'nunique')
).sort_index()

sl_export['aov'] = sl_export['revenue'] / sl_export['orders']

# Price segments
price_segments = sl.groupby(['Year Week', 'price_category'])['price'].sum().unstack()
price_segments.columns = ['revenue_high', 'revenue_low', 'revenue_mid']
sl_export = sl_export.join(price_segments)

# Seller status (returning vs new)
pre_s = sl[sl['Year Week'] < '2018-07']
dur_s = sl[(sl['Year Week'] >= '2018-07') & (sl['Year Week'] <= '2018-26')]

pre_seller_ids = set(pre_s['seller_id'].unique())
dur_seller_ids = set(dur_s['seller_id'].unique())
returning_sellers = pre_seller_ids & dur_seller_ids
new_sellers       = dur_seller_ids - pre_seller_ids

def seller_status(seller_id):
    if seller_id in returning_sellers: return 'returning'
    elif seller_id in (pre_seller_ids - dur_seller_ids): return 'churned'
    else: return 'new'

sl['seller_status'] = sl['seller_id'].apply(seller_status)

seller_weekly = sl.groupby(['Year Week', 'seller_status'])['price'].sum().unstack()
seller_weekly = seller_weekly[['new', 'returning']]
seller_weekly.columns = ['revenue_new_sellers', 'revenue_returning_sellers']
sl_export = sl_export.join(seller_weekly)

# 6-week rolling averages
cols_to_roll = [
    'revenue', 'orders', 'aov', 'avg_review', 'avg_delivery',
    'avg_freight_ratio', 'unique_customers', 'unique_sellers',
    'revenue_high', 'revenue_low', 'revenue_mid',
    'revenue_new_sellers', 'revenue_returning_sellers'
]
for col in cols_to_roll:
    sl_export[f'{col}_6w'] = sl_export[col].rolling(window=6).mean()
# Add Sort Week
sl_export['Sort Week'] = sl_export.index.str.replace('-', '').astype(int)

sl_export.to_csv('sports_leisure_weekly.csv', index=True, decimal='.', sep=',')

print('Exported: sports_leisure_weekly.csv')
sl_export.loc['2018-07':'2018-26'].head()

Price thresholds: low <= nan, mid <= nan, high > nan


ValueError: Length mismatch: Expected axis has 0 elements, new values have 3 elements

In [14]:
print(sl['price_category'].value_counts())
print(price_segments)

Series([], Name: count, dtype: int64)
Empty DataFrame
Columns: []
Index: []


In [ ]:
print(sl_export.columns.tolist())

['revenue', 'orders', 'avg_review', 'avg_delivery', 'avg_freight_ratio', 'unique_customers', 'unique_sellers', 'aov', 'revenue_high', 'revenue_low', 'revenue_mid', 'revenue_new_sellers', 'revenue_returning_sellers', 'revenue_6w', 'orders_6w', 'aov_6w', 'avg_review_6w', 'avg_delivery_6w', 'avg_freight_ratio_6w', 'unique_customers_6w', 'unique_sellers_6w', 'revenue_high_6w', 'revenue_low_6w', 'revenue_mid_6w', 'revenue_new_sellers_6w', 'revenue_returning_sellers_6w', 'Sort Week']


## 6. Revenue & Orders — Decline Overview

In [16]:
sl_rolling = sl_export.rolling(window=6).mean()
sl_plot    = sl_rolling.loc['2018-07':'2018-26'].copy()
labels     = sl_plot.index.astype(str).tolist()
x_pos      = list(range(len(labels)))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_pos, y=sl_plot['revenue'],
    mode='lines', name='Revenue (R$)',
    line=dict(color='#E24B4A', width=2), yaxis='y1'
))
fig.add_trace(go.Scatter(
    x=x_pos, y=sl_plot['orders'],
    mode='lines', name='Orders',
    line=dict(color='#60aaff', width=2, dash='dash'), yaxis='y2'
))

fig.update_layout(
    template='plotly_dark',
    title=dict(text='Revenue and Orders Declined Together (6w Moving Average)<br><sup>Sports & Leisure · 2018-07 – 2018-26</sup>', font=dict(size=14)),
    height=400,
    margin=dict(l=60, r=60, t=80, b=80),
    legend=dict(orientation='h', y=1.15, x=0, bgcolor='rgba(0,0,0,0)'),
    yaxis=dict(range=[8000, 22000], dtick=2000, tickfont=dict(color='#E24B4A'), tickprefix='R$'),
    yaxis2=dict(overlaying='y', side='right', range=[80, 170], dtick=20, showgrid=False, tickfont=dict(color='#60aaff')),
    xaxis=dict(title=dict(text='Week', standoff=10), tickvals=x_pos, ticktext=labels, ticklabelstandoff=10),
    plot_bgcolor='#1c1c2e', paper_bgcolor='#1c1c2e'
)
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## 7. Price Segment Analysis — High Price Products Drove the Decline

In [17]:
sl_plot = sl_rolling.loc['2018-07':'2018-26']
labels  = sl_plot.index.astype(str).tolist()
x_pos   = list(range(len(labels)))

fig = go.Figure()

for col, name, color in [
    ('revenue_high', 'High (>R$110)',     '#534AB7'),
    ('revenue_mid',  'Mid (R$55–110)',    '#1D9E75'),
    ('revenue_low',  'Low (<R$55)',       '#888780')
]:
    fig.add_trace(go.Scatter(
        x=x_pos, y=sl_plot[col],
        mode='lines', name=name,
        line=dict(color=color, width=2)
    ))

fig.update_layout(
    template='plotly_dark',
    title=dict(text='High-Price Products Drove the Decline (6w Moving Average)<br><sup>High segment lost 57% — Mid and Low held better</sup>', font=dict(size=14)),
    height=400,
    margin=dict(l=60, r=60, t=80, b=80),
    legend=dict(orientation='h', y=1.15, x=0, bgcolor='rgba(0,0,0,0)'),
    yaxis=dict(tickprefix='R$'),
    xaxis=dict(title='Week', tickvals=x_pos, ticktext=labels, tickangle=45),
    plot_bgcolor='#1c1c2e', paper_bgcolor='#1c1c2e'
)
fig.show()

KeyError: 'revenue_high'

## 8. Seller Analysis — Active Sellers Declined, Not Just Churned

In [ ]:
sl_plot = sl_rolling.loc['2018-07':'2018-26']
labels  = sl_plot.index.astype(str).tolist()
x_pos   = list(range(len(labels)))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_pos, y=sl_plot['revenue_returning_sellers'],
    mode='lines', name='Returning Sellers',
    line=dict(color='#534AB7', width=2)
))
fig.add_trace(go.Scatter(
    x=x_pos, y=sl_plot['revenue_new_sellers'],
    mode='lines', name='New Sellers',
    line=dict(color='#1D9E75', width=2, dash='dash')
))

fig.update_layout(
    template='plotly_dark',
    title=dict(text='Active Sellers Lost Revenue — New Sellers Could Not Compensate<br><sup>Returning sellers drove 67% of revenue loss</sup>', font=dict(size=14)),
    height=400,
    margin=dict(l=60, r=60, t=80, b=80),
    legend=dict(orientation='h', y=1.15, x=0, bgcolor='rgba(0,0,0,0)'),
    yaxis=dict(tickprefix='R$'),
    xaxis=dict(title='Week', tickvals=x_pos, ticktext=labels, tickangle=45),
    plot_bgcolor='#1c1c2e', paper_bgcolor='#1c1c2e'
)
fig.show()

## 9. Seller Scatter — Who Grew vs Who Declined

In [ ]:
pre_scatter = sl[(sl['Year Week'] >= '2017-39') & (sl['Year Week'] < '2018-07')]
dur_scatter = sl[(sl['Year Week'] >= '2018-07') & (sl['Year Week'] <= '2018-26')]

seller_scatter = pd.DataFrame({
    'pre_avg': pre_scatter.groupby('seller_id')['price'].mean(),
    'dur_avg': dur_scatter.groupby('seller_id')['price'].mean(),
}).dropna()

seller_scatter['status'] = seller_scatter.apply(
    lambda x: 'Grew' if x['dur_avg'] > x['pre_avg'] else 'Declined', axis=1
)

seller_scatter.index.name = 'seller_id'
seller_scatter.to_csv('seller_scatter.csv', index=True, header=True)

print(seller_scatter['status'].value_counts())
seller_scatter.head()
seller_scatter.columns

status
Declined    88
Grew        82
Name: count, dtype: int64


Index(['pre_avg', 'dur_avg', 'status'], dtype='str')

## 10. Customer Analysis — 89% Are One-Time Buyers

In [18]:
# Repeat vs one-time buyers
repeat_customers = sl.groupby('customer_unique_id')['order_id'].count()
total = len(repeat_customers)
once  = (repeat_customers == 1).sum()
more  = (repeat_customers > 1).sum()

print(f'Total unique customers: {total}')
print(f'Bought once:            {once} ({once/total*100:.1f}%)')
print(f'Bought more than once:  {more} ({more/total*100:.1f}%)')

# Weekly unique customers during decline
customers_weekly = sl.groupby('Year Week')['customer_unique_id'].nunique()
customers_rolling = customers_weekly.rolling(window=6).mean()
customers_plot = customers_rolling.loc['2018-07':'2018-26']

labels = customers_plot.index.astype(str).tolist()
x_pos  = list(range(len(labels)))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x_pos, y=customers_plot.values,
    mode='lines+markers', name='Unique Customers',
    line=dict(color='#378ADD', width=2), marker=dict(size=4)
))

fig.update_layout(
    template='plotly_dark',
    title=dict(text='New Customer Acquisition Slowed During Decline<br><sup>89% one-time buyers — no loyal base to compensate</sup>', font=dict(size=14)),
    height=400,
    margin=dict(l=60, r=60, t=80, b=80),
    xaxis=dict(title='Week', tickvals=x_pos, ticktext=labels, tickangle=45),
    yaxis=dict(title='Unique Customers'),
    plot_bgcolor='#1c1c2e', paper_bgcolor='#1c1c2e'
)
fig.show()

Total unique customers: 0
Bought once:            0 (nan%)
Bought more than once:  0 (nan%)


C:\Users\jonas\AppData\Local\Temp\ipykernel_20676\2113107619.py:8: RuntimeWarning: invalid value encountered in scalar divide
  print(f'Bought once:            {once} ({once/total*100:.1f}%)')
C:\Users\jonas\AppData\Local\Temp\ipykernel_20676\2113107619.py:9: RuntimeWarning: invalid value encountered in scalar divide
  print(f'Bought more than once:  {more} ({more/total*100:.1f}%)')


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## 11. Why Delivery Time and Reviews Were NOT the Cause

In [ ]:
sl_plot = sl_rolling.loc['2018-07':'2018-26']
labels  = sl_plot.index.astype(str).tolist()
x_pos   = list(range(len(labels)))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_pos, y=sl_plot['avg_review'],
    mode='lines', name='Avg Review Score',
    line=dict(color='#1D9E75', width=2)
))
fig.add_trace(go.Scatter(
    x=x_pos, y=sl_plot['avg_delivery'],
    mode='lines', name='Avg Delivery Time (days)',
    line=dict(color='#378ADD', width=2, dash='dash'), yaxis='y2'
))

fig.update_layout(
    template='plotly_dark',
    title=dict(text='Reviews Improved, Delivery Got Faster — Not the Cause of Decline<br><sup>Quality metrics moved in the right direction during revenue drop</sup>', font=dict(size=14)),
    height=400,
    margin=dict(l=60, r=60, t=80, b=80),
    legend=dict(orientation='h', y=1.15, x=0, bgcolor='rgba(0,0,0,0)'),
    yaxis=dict(title='Review Score', range=[3.5, 4.6], tickfont=dict(color='#1D9E75')),
    yaxis2=dict(title='Delivery Days', overlaying='y', side='right', showgrid=False, tickfont=dict(color='#378ADD')),
    xaxis=dict(title='Week', tickvals=x_pos, ticktext=labels, tickangle=45),
    plot_bgcolor='#1c1c2e', paper_bgcolor='#1c1c2e'
)
fig.show()